# 15 — Trajectory Profile Card

The pipeline's product is a **semantic profile per trajectory**, not a recommendation. Notebooks 1–9 plus the SR-* rules enrich every trajectory with ~12 tag types stored in the KG. This notebook surfaces them.

For a given trajectory URI, this notebook runs one SPARQL per field and emits a structured card:

- **Duration** — start, end, hours, position count
- **Purpose** (`trajecto:hasPurpose`) — Cultural / Dining / Shopping / …
- **Navigation style** (`patterns:exhibitsPattern`) — Casual / Targeted / Immersive / Exploratory
- **Engagement level** (`trajecto:hasEngagementLevel`) — low / medium / high
- **Temporal context** (`context:hasTemporalContext`) — time-of-day / weekday / season / holiday
- **Dominant POI** (`poi:DominantInteraction`) — singleton per trajectory (enforced by SHACL `ex:DominantSingletonShape`)
- **Significant POIs** (`rdf:type poi:SignificantPOI`)
- **Notable non-transport POIs** — meaningful+ interactions filtered to non-transport categories, ranked by interaction count. Surfaces cultural / dining / leisure POIs when the singleton dominant is a transport POI.
- **Category arc** — ordered sequence of POI categories (transport/navigation categories filtered out)
- **Interaction counts (disjoint)** — exact-type counts via most-specific type classification. A naive `COUNT` would double-count because `Immersive ⊂ Meaningful` and `Dominant ⊂ Meaningful` in the ontology.
- **Co-visited trajectories** (SR-05 / `poi:covisitedWith`)
- **Succession prediction** (`poi:followedBy`) — likely next category
- **User preferences** (`user:hasPreferences`) — top-3 categories

Downstream:
- Notebook 16 — consumer demo: unified recommender with 4-layer explainability + map of visited + recommended.
- Notebook 17 — headline evaluation: recommender vs baseline ladder + proxy metrics.
- Notebook 18 — diagnostics: signal weight sensitivity + drop-one ablation.

In [ ]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd
from SPARQLWrapper import JSON, SPARQLWrapper

from helpers.fuseki_config import query_endpoint

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 180)

PREFIXES = '''
PREFIX trajecto: <http://w3id.org/trajecto#>
PREFIX poi:      <http://w3id.org/trajecto/poi#>
PREFIX patterns: <http://w3id.org/trajecto/patterns#>
PREFIX context:  <http://w3id.org/trajecto/context#>
PREFIX user:     <http://w3id.org/trajecto/user#>
PREFIX purpose:  <http://w3id.org/trajecto/purpose#>
PREFIX datacron: <http://www.datacron-project.eu/datAcron#>
PREFIX dul:      <http://www.ontologydesignpatterns.org/ont/dul/DUL.owl#>
PREFIX rdfs:     <http://www.w3.org/2000/01/rdf-schema#>
PREFIX rdf:      <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
'''

# Mirrors EXCLUDED_CATEGORIES in context_recommender.py. Used to filter transport/navigation/utility
# POIs out of dominant / notable / arc surfacing so cultural/dining/leisure POIs arise.
EXCLUDED_CATEGORIES = {
    'ATM', 'Bank', 'PostOffice',
    'BusStation', 'TrainStation', 'Airport', 'Transportation', 'TransportationPOI',
    'TransitStop', 'ParkingArea',
    'Elevator', 'Escalator', 'Staircase',
    'Entrance', 'Exit', 'Intersection', 'Junction',
    'Transition', 'TransitionPOI', 'Waypoint', 'NavigationLandmark',
    'Restroom', 'RestRoom', 'ServicePOI',
    'PointofInterest', 'PointOfInterest', 'SignificantPOI',
    'nan', 'None', 'Unknown', 'Uncategorized', '',
}


def sparql(q):
    s = SPARQLWrapper(query_endpoint)
    s.setQuery(PREFIXES + q)
    s.setReturnFormat(JSON)
    res = s.query().convert()
    rows = []
    for b in res['results']['bindings']:
        rows.append({k: v['value'] for k, v in b.items()})
    return pd.DataFrame(rows)


def short(uri):
    if not isinstance(uri, str) or not uri:
        return ''
    return uri.split('#')[-1].split('/')[-1]

print(f'Endpoint: {query_endpoint}')


## 1. Per-field SPARQL queries

Each function below queries one field of the card. All accept a trajectory URI string and return a scalar, list, or dict.

`tag_interaction_counts` uses `FILTER NOT EXISTS` to enforce **most-specific-type** counting: `Immersive ⊂ Meaningful` and `Dominant ⊂ Meaningful` in the ontology, so a naive `COUNT` would treat each immersive/dominant as also meaningful and double-count.

`tag_dominant_poi` assumes exactly one `DominantInteraction` per trajectory after the NB 5 fix and the SHACL shape `ex:DominantSingletonShape`. If the fix has not been re-run on an existing dataset, stale data can still produce multiple; the function picks one deterministically.

`tag_notable_pois` exists because a trajectory can have a transport-category dominant POI (bus station, train station) while its interesting POIs (cafes, landmarks, leisure) only receive meaningful interactions. This field surfaces those.


In [ ]:
def tag_duration(traj):
    df = sparql(f'''
        SELECT (MIN(?ts) AS ?start) (MAX(?ts) AS ?end) (COUNT(?pos) AS ?nPos)
        WHERE {{
            ?pos dul:isPartOf <{traj}> ;
                 trajecto:timestamp ?ts .
        }}
    ''')
    if df.empty or 'start' not in df.columns or not df['start'].iloc[0]:
        return None
    start = pd.to_datetime(df['start'].iloc[0], errors='coerce')
    end = pd.to_datetime(df['end'].iloc[0], errors='coerce')
    if pd.isna(start) or pd.isna(end):
        return None
    hours = (end - start).total_seconds() / 3600.0
    return {
        'start': str(start),
        'end': str(end),
        'hours': round(hours, 2),
        'n_positions': int(df['nPos'].iloc[0]) if 'nPos' in df.columns and df['nPos'].iloc[0] else 0,
    }


def tag_purpose(traj):
    df = sparql(f'''
        SELECT ?purpose WHERE {{
            <{traj}> trajecto:hasPurpose ?purpose .
        }}
    ''')
    return [short(p) for p in df['purpose'].tolist()] if not df.empty else []


def tag_nav_style(traj):
    df = sparql(f'''
        SELECT ?style WHERE {{
            <{traj}> patterns:exhibitsPattern ?style .
        }}
    ''')
    return [short(s) for s in df['style'].tolist()] if not df.empty else []


def tag_engagement(traj):
    df = sparql(f'''
        SELECT ?level WHERE {{
            <{traj}> trajecto:hasEngagementLevel ?level .
        }}
    ''')
    return short(df['level'].iloc[0]) if not df.empty else None


def tag_temporal_context(traj):
    df = sparql(f'''
        SELECT DISTINCT ?ctx WHERE {{
            <{traj}> context:hasTemporalContext ?ctx .
        }}
    ''')
    return sorted({short(c) for c in df['ctx'].tolist()}) if not df.empty else []


# ----------------------------------------------------------------------
# The tag_* functions below share a pattern: first resolve the distinct
# interactions / POIs for the trajectory via a subquery, then join the
# secondary facts. Without the DISTINCT subquery the outer join fans out
# by position count (200+ per interaction), producing ~N² row explosions
# that made build_card look like it was hanging.
# ----------------------------------------------------------------------


def tag_dominant_poi(traj):
    df = sparql(f'''
        SELECT DISTINCT ?int ?poi ?label ?cat WHERE {{
            {{
                SELECT DISTINCT ?int WHERE {{
                    ?pos dul:isPartOf <{traj}> ; trajecto:hasInteraction ?int .
                    ?int a poi:DominantInteraction .
                }}
            }}
            ?int poi:interactsWith ?poi .
            OPTIONAL {{ ?poi rdfs:label ?label . }}
            OPTIONAL {{
                ?poi a ?cat .
                FILTER(?cat != trajecto:PointOfInterest && ?cat != poi:SignificantPOI)
            }}
        }} ORDER BY ?int LIMIT 1
    ''')
    if df.empty:
        return None
    row = df.iloc[0]
    return {
        'poi': short(row.get('poi', '')),
        'label': row.get('label') or short(row.get('poi', '')),
        'category': short(row.get('cat', '')),
    }


def tag_significant_pois(traj):
    df = sparql(f'''
        SELECT DISTINCT ?poi ?label WHERE {{
            {{
                SELECT DISTINCT ?poi WHERE {{
                    ?pos dul:isPartOf <{traj}> ; trajecto:hasInteraction ?int .
                    ?int poi:interactsWith ?poi .
                }}
            }}
            ?poi a poi:SignificantPOI .
            OPTIONAL {{ ?poi rdfs:label ?label . }}
        }}
    ''')
    if df.empty:
        return []
    return [{'poi': short(r['poi']), 'label': r.get('label') or short(r['poi'])}
            for _, r in df.iterrows()]


def tag_notable_pois(traj, k=8):
    df = sparql(f'''
        SELECT ?poi ?label ?cat (COUNT(DISTINCT ?int) AS ?n) WHERE {{
            {{
                SELECT DISTINCT ?int WHERE {{
                    ?pos dul:isPartOf <{traj}> ; trajecto:hasInteraction ?int .
                    ?int a poi:MeaningfulInteraction .
                }}
            }}
            ?int poi:interactsWith ?poi .
            OPTIONAL {{ ?poi rdfs:label ?label . }}
            OPTIONAL {{
                ?poi a ?cat .
                FILTER(?cat != trajecto:PointOfInterest && ?cat != poi:SignificantPOI)
            }}
        }} GROUP BY ?poi ?label ?cat
           ORDER BY DESC(?n)
    ''')
    if df.empty:
        return []
    out, seen = [], set()
    for _, r in df.iterrows():
        cat = short(r.get('cat', ''))
        if cat in EXCLUDED_CATEGORIES:
            continue
        poi = short(r['poi'])
        if poi in seen:
            continue
        seen.add(poi)
        out.append({
            'poi': poi,
            'label': r.get('label') or poi,
            'category': cat,
            'interactions': int(r['n']),
        })
        if len(out) >= k:
            break
    return out


def tag_category_arc(traj):
    df = sparql(f'''
        SELECT ?int ?cat ?pos WHERE {{
            {{
                SELECT ?int (MIN(?p) AS ?pos) WHERE {{
                    ?p dul:isPartOf <{traj}> ; trajecto:hasInteraction ?int .
                    ?int a poi:MeaningfulInteraction .
                }} GROUP BY ?int
            }}
            ?int poi:interactsWith ?poi .
            OPTIONAL {{
                ?poi a ?cat .
                FILTER(?cat != trajecto:PointOfInterest && ?cat != poi:SignificantPOI)
            }}
        }} ORDER BY ?pos
    ''')
    if df.empty:
        return []
    seq, last, seen_int = [], None, set()
    for _, r in df.iterrows():
        if r['int'] in seen_int:
            continue
        seen_int.add(r['int'])
        c = short(r.get('cat', ''))
        if not c or c in EXCLUDED_CATEGORIES:
            continue
        if c != last:
            seq.append(c)
            last = c
    return seq


def tag_interaction_counts(traj):
    """Fetch (interaction, type) pairs for the trajectory's interactions in one flat
    query, then classify into most-specific type in Python. Avoids N UNION branches
    each doing its own FILTER NOT EXISTS scan."""
    df = sparql(f'''
        SELECT DISTINCT ?int ?type WHERE {{
            {{
                SELECT DISTINCT ?int WHERE {{
                    ?pos dul:isPartOf <{traj}> ; trajecto:hasInteraction ?int .
                }}
            }}
            ?int a ?type .
            FILTER(?type IN (poi:MeaningfulInteraction,
                             poi:ImmersiveInteraction,
                             poi:DominantInteraction,
                             poi:StartInteraction,
                             poi:DestinationInteraction))
        }}
    ''')
    if df.empty:
        return {}
    types_by_int = {}
    for _, r in df.iterrows():
        types_by_int.setdefault(r['int'], set()).add(short(r['type']))
    counts = Counter()
    for types in types_by_int.values():
        if 'DominantInteraction' in types:
            counts['Dominant'] += 1
        elif 'ImmersiveInteraction' in types:
            counts['Immersive'] += 1
        elif 'MeaningfulInteraction' in types:
            counts['Meaningful'] += 1
        if 'StartInteraction' in types:
            counts['Start'] += 1
        if 'DestinationInteraction' in types:
            counts['Destination'] += 1
    out = {k: int(counts.get(k, 0)) for k in
           ['Meaningful', 'Immersive', 'Dominant', 'Start', 'Destination']}
    out['total_distinct'] = sum(out.values())
    return out


def tag_covisited_trajectories(traj, k=5, poi_limit=3):
    """Biggest speed hazard in the card. The naive position × position self-join is
    ~N² over positions (10k+ positions per trajectory pair). Two DISTINCT subqueries
    shrink the cross-product to POIs × other-trajectories."""
    df = sparql(f'''
        SELECT DISTINCT ?other ?poi ?label ?cat WHERE {{
            {{
                SELECT DISTINCT ?poi WHERE {{
                    ?pos dul:isPartOf <{traj}> ; trajecto:hasInteraction ?int .
                    ?int poi:interactsWith ?poi .
                }}
            }}
            {{
                SELECT DISTINCT ?poi ?other WHERE {{
                    ?pos2 dul:isPartOf ?other ; trajecto:hasInteraction ?int2 .
                    ?int2 poi:interactsWith ?poi .
                }}
            }}
            FILTER(?other != <{traj}>)
            OPTIONAL {{ ?poi rdfs:label ?label . }}
            OPTIONAL {{
                ?poi a ?cat .
                FILTER(?cat != trajecto:PointOfInterest && ?cat != poi:SignificantPOI)
            }}
        }}
    ''')
    if df.empty:
        return []

    df['cat_short'] = df.get('cat', pd.Series(dtype=object)).fillna('').map(short)
    bad_pois = set(df.loc[df['cat_short'].isin(EXCLUDED_CATEGORIES), 'poi'])
    df = df[~df['poi'].isin(bad_pois)]
    if df.empty:
        return []

    poi_info = {}
    for _, r in df.iterrows():
        p = r['poi']
        info = poi_info.setdefault(p, {'poi': short(p), 'label': short(p), 'category': ''})
        if r.get('label') and info['label'] == short(p):
            info['label'] = r['label']
        if r['cat_short'] and not info['category']:
            info['category'] = r['cat_short']

    grouped = {}
    for _, r in df.iterrows():
        grouped.setdefault(r['other'], set()).add(r['poi'])

    rows = []
    for other, pois in grouped.items():
        shared = [poi_info[p] for p in pois]
        shared.sort(key=lambda x: x['label'])
        rows.append({
            'trajectory': short(other),
            'shared_pois': shared[:poi_limit],
            'shared_total': len(pois),
        })
    rows.sort(key=lambda r: -r['shared_total'])
    return rows[:k]


def tag_succession_prediction(traj, k=3):
    df = sparql(f'''
        SELECT ?nextCat (COUNT(?next) AS ?n) WHERE {{
            {{
                SELECT DISTINCT ?src WHERE {{
                    ?pos dul:isPartOf <{traj}> ; trajecto:hasInteraction ?int .
                    ?int poi:interactsWith ?src .
                }}
            }}
            ?src poi:followedBy ?next .
            OPTIONAL {{
                ?next a ?nextCat .
                FILTER(?nextCat != trajecto:PointOfInterest && ?nextCat != poi:SignificantPOI)
            }}
        }} GROUP BY ?nextCat
           ORDER BY DESC(?n)
           LIMIT {k}
    ''')
    if df.empty:
        return []
    rows = [(short(r['nextCat']), int(r['n'])) for _, r in df.iterrows() if r.get('nextCat')]
    total = sum(n for _, n in rows) or 1
    return [{'category': c, 'likelihood': round(n/total, 3)} for c, n in rows]


def tag_user_preferences(traj, k=3):
    df = sparql(f'''
        SELECT ?category ?score ?confidence WHERE {{
            ?user trajecto:formsTrajectory <{traj}> .
            ?user user:hasProfile ?profile .
            ?profile user:hasSnapshot ?snap .
            ?snap user:hasPreferences ?pref .
            ?pref user:poiCategory ?category ;
                  user:preferenceScore ?score ;
                  user:confidenceLevel ?confidence .
            ?snap user:timestamp ?ts .
        }} ORDER BY DESC(?ts) DESC(?score) LIMIT {k}
    ''')
    if df.empty:
        return []
    return [{'category': short(r['category']),
             'score': round(float(r['score']), 3),
             'confidence': round(float(r['confidence']), 3)}
            for _, r in df.iterrows()]


## 2. Card assembly and rendering

`build_card(traj_uri)` returns a plain dict. `render_card(card)` formats it as markdown for display or export.

In [ ]:
def build_card(traj):
    return {
        'trajectory': short(traj),
        'uri': traj,
        'duration': tag_duration(traj),
        'purpose': tag_purpose(traj),
        'navigation_style': tag_nav_style(traj),
        'engagement': tag_engagement(traj),
        'temporal_context': tag_temporal_context(traj),
        'dominant_poi': tag_dominant_poi(traj),
        'significant_pois': tag_significant_pois(traj),
        'notable_pois': tag_notable_pois(traj),
        'category_arc': tag_category_arc(traj),
        'interaction_counts': tag_interaction_counts(traj),
        'covisited_trajectories': tag_covisited_trajectories(traj),
        'succession_prediction': tag_succession_prediction(traj),
        'user_preferences': tag_user_preferences(traj),
    }


def render_card(card):
    lines = [f"## Trajectory {card['trajectory']}", '']
    d = card['duration']
    if d:
        lines.append(f"**Duration:**          {d['hours']}h ({d['n_positions']} positions)  {d['start']} → {d['end']}")
    else:
        lines.append("**Duration:**          —")
    lines.append(f"**Purpose:**           {', '.join(card['purpose']) or '—'}")
    lines.append(f"**Navigation style:**  {', '.join(card['navigation_style']) or '—'}")
    lines.append(f"**Engagement:**        {card['engagement'] or '—'}")
    lines.append(f"**Temporal context:**  {', '.join(card['temporal_context']) or '—'}")
    dom = card['dominant_poi']
    lines.append(f"**Dominant POI:**      {dom['label'] + ' (' + dom['category'] + ')' if dom else '—'}")
    sig = card['significant_pois']
    if sig:
        labels = ', '.join(p['label'] for p in sig[:6])
        more = f' (+{len(sig)-6} more)' if len(sig) > 6 else ''
        lines.append(f"**Significant POIs:**  {labels}{more}")
    else:
        lines.append("**Significant POIs:**  —")
    if card['notable_pois']:
        notable = ', '.join(f"{p['label']} ({p['category']}, n={p['interactions']})" for p in card['notable_pois'])
        lines.append(f"**Notable POIs:**      {notable}")
    else:
        lines.append("**Notable POIs:**      —")
    lines.append(f"**Category arc:**      {' → '.join(card['category_arc']) or '—'}")
    ic = card['interaction_counts']
    if ic:
        keys = ['Meaningful', 'Immersive', 'Dominant', 'Start', 'Destination']
        ic_str = ', '.join(f'{k}={ic.get(k,0)}' for k in keys) + f" (total_distinct={ic.get('total_distinct',0)})"
    else:
        ic_str = '—'
    lines.append(f"**Interactions:**      {ic_str}")
    cov = card['covisited_trajectories']
    if cov:
        parts = []
        for c in cov[:3]:
            pois_str = ', '.join(f"{p['label']}/{p['category']}" if p['category'] else p['label']
                                 for p in c['shared_pois'])
            extra = c['shared_total'] - len(c['shared_pois'])
            more = f' +{extra} more' if extra > 0 else ''
            parts.append(f"{c['trajectory']} ({pois_str}{more})")
        lines.append("**Co-visited with:**   " + '; '.join(parts))
    else:
        lines.append("**Co-visited with:**   —")
    suc = card['succession_prediction']
    if suc:
        lines.append("**Likely next:**       " + ', '.join(f"{s['category']}({s['likelihood']})" for s in suc))
    else:
        lines.append("**Likely next:**       —")
    pref = card['user_preferences']
    if pref:
        lines.append("**User preferences:**  " + ', '.join(f"{p['category']}({p['score']})" for p in pref))
    else:
        lines.append("**User preferences:**  —")
    return '\n'.join(lines)


## 3. Pick trajectories and render one

Sample trajectories that have enough structure to populate the card (≥2 meaningful-or-deeper interactions).

In [ ]:
eligible = sparql('''
    SELECT ?traj (COUNT(DISTINCT ?poi) AS ?n) WHERE {
        ?pos dul:isPartOf ?traj ;
             trajecto:hasInteraction ?int .
        ?int a ?itype ;
             poi:interactsWith ?poi .
        FILTER(?itype IN (poi:MeaningfulInteraction,
                          poi:ImmersiveInteraction,
                          poi:DominantInteraction))
    } GROUP BY ?traj HAVING (?n >= 2)
      ORDER BY DESC(?n) LIMIT 20
''')
print(f'Eligible trajectories (≥2 POIs): {len(eligible)}')
eligible.head()


In [ ]:
if not eligible.empty:
    card = build_card(eligible['traj'].iloc[2])
    print(render_card(card))
else:
    print('No eligible trajectories found.')


## 3b. Map — actual trajectory + visited POIs

Same trajectory as the card, rendered on a Folium map:

- **Solid line** — actual GPS trajectory (RawPosition sequence ordered by index / timestamp)
- **Numbered markers** — visited POIs in sequence order (by first interaction position)
- **Dashed line** — POI-to-POI transitions (implied order of visited POIs)
- **Popup** — POI label, category, dwell time, interaction type

Product-surface only. Recommendations live downstream in notebook 16.

In [ ]:
import folium
from IPython.display import display as ipydisplay
from helpers.map_visualizer import TrajectoMapVisualizer

visualizer = TrajectoMapVisualizer()


def fetch_trajectory_path(traj):
    """Pull the actual GPS trace: every RawPosition's geometry, ordered by
    position index (fallback to timestamp). Returns list of [lat, lon]."""
    df = sparql(f'''
        SELECT ?pos ?lat ?lon ?idx ?ts WHERE {{
            ?pos dul:isPartOf <{traj}> ;
                 datacron:hasGeometry ?geom .
            ?geom datacron:hasLatitude ?lat ;
                  datacron:hasLongitude ?lon .
            OPTIONAL {{ ?pos trajecto:hasPositionIndex ?idx . }}
            OPTIONAL {{ ?pos trajecto:timestamp ?ts . }}
        }}
    ''')
    if df.empty:
        return []
    df['_idx'] = pd.to_numeric(df.get('idx'), errors='coerce')
    df['_ts'] = pd.to_numeric(df.get('ts'), errors='coerce')
    df = df.sort_values(['_idx', '_ts'], na_position='last')
    return [[float(r['lat']), float(r['lon'])] for _, r in df.iterrows()
            if r.get('lat') and r.get('lon')]


def fetch_visited_pois(traj):
    """One SPARQL, DISTINCT on interaction. Order by first position per interaction."""
    df = sparql(f'''
        SELECT ?int ?poi ?label ?cat ?dwell ?itype (MIN(?pIdx) AS ?firstIdx) WHERE {{
            ?pos dul:isPartOf <{traj}> ;
                 trajecto:hasInteraction ?int .
            ?int poi:interactsWith ?poi ;
                 a ?itype .
            FILTER(?itype IN (poi:MeaningfulInteraction, poi:ImmersiveInteraction,
                              poi:DominantInteraction,
                              poi:StartInteraction, poi:DestinationInteraction))
            OPTIONAL {{ ?pos trajecto:hasPositionIndex ?pIdx . }}
            OPTIONAL {{ ?poi rdfs:label ?label . }}
            OPTIONAL {{ ?poi datacron:hasLatitude ?lat . }}
            OPTIONAL {{ ?poi datacron:hasLongitude ?lon . }}
            OPTIONAL {{ ?int poi:hasDwellTimeInSeconds ?dwell . }}
            OPTIONAL {{
                ?poi a ?cat .
                FILTER(?cat != trajecto:PointOfInterest && ?cat != poi:SignificantPOI)
            }}
        }} GROUP BY ?int ?poi ?label ?cat ?dwell ?itype
    ''')
    # Need coords separately to avoid group-by blowup
    coords = sparql(f'''
        SELECT DISTINCT ?poi ?lat ?lon WHERE {{
            {{
                SELECT DISTINCT ?poi WHERE {{
                    ?pos dul:isPartOf <{traj}> ; trajecto:hasInteraction ?int .
                    ?int poi:interactsWith ?poi .
                }}
            }}
            ?poi datacron:hasLatitude ?lat ;
                 datacron:hasLongitude ?lon .
        }}
    ''')
    coord_map = {r['poi']: (float(r['lat']), float(r['lon']))
                 for _, r in coords.iterrows() if r.get('lat') and r.get('lon')}

    # Rank interaction type specificity so Dominant > Immersive > Meaningful > Start/Dest
    rank = {'DominantInteraction': 4, 'ImmersiveInteraction': 3,
            'MeaningfulInteraction': 2, 'StartInteraction': 1, 'DestinationInteraction': 1}

    # One row per interaction; keep the most specific type
    best = {}
    for _, r in df.iterrows():
        key = r['int']
        rk = rank.get(short(r.get('itype', '')), 0)
        prev = best.get(key)
        if prev is None or rk > prev['_rank']:
            best[key] = {**r.to_dict(), '_rank': rk}

    rows = []
    for _, info in best.items():
        poi = info['poi']
        if poi not in coord_map:
            continue
        cat = short(info.get('cat', ''))
        if cat in EXCLUDED_CATEGORIES:
            continue
        lat, lon = coord_map[poi]
        dwell = info.get('dwell')
        try:
            dwell = float(dwell) if dwell else 0.0
        except (ValueError, TypeError):
            dwell = 0.0
        rows.append({
            'poi': poi,
            'label': info.get('label') or short(poi),
            'latitude': lat,
            'longitude': lon,
            'category': cat,
            'dwell_seconds': dwell,
            'interaction_type': short(info.get('itype', '')),
            'first_idx': int(info['firstIdx']) if info.get('firstIdx') else 0,
        })
    rows.sort(key=lambda r: r['first_idx'])
    return rows


MAPS_DIR = Path('output/maps')
MAPS_DIR.mkdir(parents=True, exist_ok=True)

if not eligible.empty:
    traj_uri = eligible['traj'].iloc[1]
    visited = fetch_visited_pois(traj_uri)
    path = fetch_trajectory_path(traj_uri)
    print(f'{short(traj_uri)}: {len(visited)} visited POIs, {len(path)} GPS positions')
    if visited or path:
        m = visualizer.create_trajectory_map(
            visited,
            title=f'Trajectory + visited POIs: {short(traj_uri)}',
        )
        if path:
            folium.PolyLine(
                path,
                color='#455A64',
                weight=2,
                opacity=0.8,
                tooltip=f'GPS trace ({len(path)} positions)',
            ).add_to(m)
            # Refit to the union of trajectory path + POI markers
            bounds = path + [[p['latitude'], p['longitude']] for p in visited]
            if len(bounds) > 1:
                m.fit_bounds(bounds)
        map_path = MAPS_DIR / f'nb15_profile_{short(traj_uri)}.html'
        m.save(str(map_path))
        print(f'Saved map: {map_path}')
        ipydisplay(m)
    else:
        print('No coordinates available for this trajectory.')


## 4. Batch export

Build cards for all eligible trajectories. Emit two artifacts:

- `output/profile_cards/cards.json` — machine-readable per-trajectory records
- `output/profile_cards/cards.md` — concatenated markdown report for human inspection

Also print a tag-coverage summary: percentage of trajectories that have each field populated. This is the honest measure of pipeline completeness.

In [ ]:
out_dir = Path('output/profile_cards')
out_dir.mkdir(parents=True, exist_ok=True)

cards = []
for traj in eligible['traj']:
    cards.append(build_card(traj))

(out_dir / 'cards.json').write_text(json.dumps(cards, indent=2, default=str))
(out_dir / 'cards.md').write_text('\n\n---\n\n'.join(render_card(c) for c in cards))

print(f'Wrote {len(cards)} cards to {out_dir}/')


In [ ]:
n = len(cards) or 1
coverage = {
    'duration':              sum(1 for c in cards if c['duration']),
    'purpose':               sum(1 for c in cards if c['purpose']),
    'navigation_style':      sum(1 for c in cards if c['navigation_style']),
    'engagement':            sum(1 for c in cards if c['engagement']),
    'temporal_context':      sum(1 for c in cards if c['temporal_context']),
    'dominant_poi':          sum(1 for c in cards if c['dominant_poi']),
    'significant_pois':      sum(1 for c in cards if c['significant_pois']),
    'notable_pois':          sum(1 for c in cards if c['notable_pois']),
    'category_arc':          sum(1 for c in cards if c['category_arc']),
    'interaction_counts':    sum(1 for c in cards if c['interaction_counts']),
    'covisited_trajectories':sum(1 for c in cards if c['covisited_trajectories']),
    'succession_prediction': sum(1 for c in cards if c['succession_prediction']),
    'user_preferences':      sum(1 for c in cards if c['user_preferences']),
}
cov_df = pd.DataFrame([
    {'field': k, 'populated': v, 'coverage_%': round(100 * v / n, 1)}
    for k, v in coverage.items()
]).sort_values('coverage_%', ascending=False)
cov_df


## 5. Reading the coverage table

Each row is one tag type the pipeline produces. The `coverage_%` column is the true measure of whether the tagging stack is delivering on this corpus — it answers *"for what fraction of trajectories does this tag exist in the KG at all?"*.

Missing fields indicate pipeline gaps, not recommender weaknesses. Low coverage on `covisited_trajectories` or `succession_prediction` means SR-05 / SR-06 have not populated enough edges; low coverage on `user_preferences` means notebook 8/9 has not run for the relevant users. These are the things to fix before judging any downstream recommender.